# Polymarket Price Series Explorer
Fetch and plot historical probability curves for prediction market events.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from data.historical import search_markets, fetch_history, fetch_histories

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

## 1. Search for markets
Pass a list of keywords — returns all matching outcome/token pairs sorted by volume.

In [ ]:
keywords = ['iran', 'fed', 'trump', 'ukraine', 'tariff']
markets = search_markets(keywords, limit=20, closed=None)

# Show what we found
summary = pd.DataFrame(markets)[['question', 'outcome', 'volume', 'closed', 'end_date']]
summary['volume_M'] = (summary['volume'] / 1e6).round(2)
summary.drop(columns='volume').sort_values('volume_M', ascending=False).head(20)

## 2. Pick markets to plot
Filter to one outcome per question (Yes / first outcome) to avoid doubling up.

In [ ]:
# Keep the highest-volume outcome per question
seen, picks = set(), []
for m in sorted(markets, key=lambda x: x['volume'], reverse=True):
    q = m['question']
    if q in seen:
        continue
    seen.add(q)
    picks.append(m)
    if len(picks) == 5:
        break

for p in picks:
    print(f"  [{p['outcome']}] {p['question'][:80]}")

## 3. Fetch full price history

In [ ]:
df = fetch_histories(picks, interval='max')
print(f"{len(df):,} rows across {df['slug'].nunique()} markets")
df.head(3)

## 4. Plot each market's probability over time

In [ ]:
groups = list(df.groupby('slug'))
n = len(groups)

fig, axes = plt.subplots(n, 1, figsize=(12, 3.5 * n), sharex=False)
if n == 1:
    axes = [axes]

for ax, (slug, grp) in zip(axes, groups):
    grp = grp.sort_values('timestamp')
    outcome = grp['outcome'].iloc[0]
    question = grp['question'].iloc[0]

    ax.plot(grp['timestamp'], grp['price'], linewidth=1.5, color='steelblue')
    ax.fill_between(grp['timestamp'], grp['price'], alpha=0.15, color='steelblue')

    ax.set_ylim(-0.02, 1.02)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_ylabel('Probability')

    # Wrap long questions
    import textwrap
    title = textwrap.fill(f"{outcome}: {question}", width=90)
    ax.set_title(title, fontsize=9, loc='left')

plt.tight_layout()
plt.show()

## 5. Overlay comparison — normalized to same time axis
Useful when markets have different durations.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for slug, grp in df.groupby('slug'):
    grp = grp.sort_values('timestamp')
    label = textwrap.fill(grp['question'].iloc[0], width=55)
    ax.plot(grp['timestamp'], grp['price'], linewidth=1.5, label=label)

ax.set_ylim(-0.02, 1.02)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.tick_params(axis='x', rotation=30)
ax.grid(linestyle='--', alpha=0.3)
ax.set_ylabel('Probability')
ax.set_title('Market probabilities over time', fontsize=11)
ax.legend(fontsize=7, loc='upper left', framealpha=0.8)

plt.tight_layout()
plt.show()

## 6. Summary statistics

In [ ]:
stats = (
    df.groupby(['question', 'outcome'])['price']
    .agg(['count', 'first', 'last', 'min', 'max', 'std'])
    .rename(columns={'count': 'n_points', 'first': 'price_start', 'last': 'price_end',
                     'min': 'price_min', 'max': 'price_max', 'std': 'volatility'})
    .round(3)
    .sort_values('n_points', ascending=False)
)
stats